# 📊 Model Training & Evaluation Loss Verification Notebook

이 노트북은 `result/` 폴더에 저장되어 있는 `gru_*.pth` 베스트 모델 파일의 **Train Loss** 및 **Validation Loss**를 재계산하여 검증하고, **Gaussian Hit Loss**와 **Wing Loss** 두 가지 손실을 모두 측정합니다.
또한, 사용자가 설정하고자 했던 Wing Loss의 하이퍼파라미터 `w`와 `epsilon`이 `0.03 / 0.005`로 되어 있는지 확인하고, 해당 파라미터 기준으로도 손실을 산출하여 보여줍니다.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# experiment 폴더 내부에서 상위 폴더(루트 경로)의 모듈을 로드하기 위해 sys.path에 추가합니다.
sys.path.insert(0, str(Path('..').resolve()))

from args import Config
from model import MosquitoGRU_M2M, MosquitoGRU, WingLoss, GaussianHitLoss, CombinedLoss
from dataset import MosquitoDataset

print("✅ 라이브러리 및 프로젝트 모듈 로드 완료!")

## 🔍 1. 저장된 모델 파일 탐색 및 구조 분석

`result/` 폴더 내에 저장된 `gru_*.pth` 형식의 파일을 탐색하여 가중치(State Dict) 구조를 파악하고, 모델이 **M2O(단일 시퀀스 예측)** 모드인지, **M2M(다대다 예측)** 모드인지를 가중치 키값을 기준으로 자동 감지합니다.

In [ ]:
# 1. result 폴더에서 모델 파일 탐색
result_dir = Path('../result')
model_files = sorted(list(result_dir.glob('gru_*.pth')))

if not model_files:
    raise FileNotFoundError("result/ 폴더에 gru_*.pth 모델 파일이 존재하지 않습니다!")

best_model_path = model_files[0]
print(f"✅ 탐색된 모델 파일: {best_model_path.name}")

# 2. State Dict 로드 및 M2M/M2O 판별
state_dict = torch.load(best_model_path, map_location='cpu')

# fc_40 레이어가 있으면 M2M 모델, 그렇지 않으면 M2O 모델입니다.
is_m2m = 'fc_40.weight' in state_dict
model_mode = 'm2m' if is_m2m else 'm2o'

print(f"✅ 감지된 모델 모드: {model_mode.upper()} (is_m2m = {is_m2m})")

## ⚙️ 2. Wing Loss 파라미터 `w = 0.03 / 0.005` 설정 상태 확인

대회 혹은 모델 세팅 상에서 사용자가 적용하려 했던 Wing Loss 파라미터 `w = 0.03`, `epsilon = 0.005`가 `args.py` (Config) 내에 어떻게 선언되어 있는지 점검합니다.

> 💡 **중요 참고**: `.pth` 모델 가중치 파일에는 모델의 레이어 가중치(Weights & Biases)만 포함되며, 학습 시 활용했던 Loss 파라미터(`w`, `epsilon`)는 포함되지 않습니다. 따라서 현재 프로젝트 설정 파일(`args.py`)의 `Config` 값을 확인하고, 두 설정(`0.03` vs `Config` 기본값) 모두에 대해 손실을 계산하도록 구성했습니다.

In [ ]:
# 현재 Config 파일에 저장된 설정값 가져오기
current_w = Config.wing_w
current_eps = Config.wing_epsilon

print("=" * 60)
print("📢 [Wing Loss Hyperparameter Verification]")
print("=" * 60)
print(f"1) 사용자가 요청한 설정값 (Target) : w = 0.03,  epsilon = 0.005")
print(f"2) 현재 Config에 정의된 설정값 (Actual) : w = {current_w:.4f}, epsilon = {current_eps:.4f}")
print("-" * 60)

if np.isclose(current_w, 0.03) and np.isclose(current_eps, 0.005):
    print("✅ 결과: 현재 Config가 요청하신 w = 0.03 / epsilon = 0.005 로 정확히 설정되어 있습니다!")
else:
    print("⚠️ 결과: 현재 Config는 다르게 설정되어 있습니다 (현재 w = 0.02 로 지정됨).")
    print("   노트북 아래 셀에서 두 파라미터 버전(Target 0.03 vs Actual 0.02)에 대해 각각 손실을 연산하여 보여줍니다.")
print("=" * 60)

## 📦 3. 데이터셋 로드 및 분리 (Train/Val Split)

`train.py`에서 수행하는 방식과 동일한 난수 시드(`Config.seed = 42`)를 사용하여 데이터셋을 8:2 비율로 분리하고 로드합니다.
- **Train Set**: Sub-sequence 증강(`subseq_aug=True`)이 포함되어 있으며, `train.py`와 같은 방식으로 처리됩니다.
- **Validation Set**: Sub-sequence 증강이 미적용되어 있으며, 원본 시퀀스 크기로 구성됩니다.

In [ ]:
# Config 모드 강제 적용
Config.model_mode = model_mode
Config.use_delta = True      # 기본값
Config.use_rotation = True   # 기본값

# 경로 설정
train_dir = Path('../data/train')
train_labels_path = Path('../data/train_labels.csv')

train_files = sorted(list(train_dir.glob('TRAIN_*.csv')))
train_labels = pd.read_csv(train_labels_path)

# 80:20 분리
train_files_split, val_files_split = train_test_split(train_files, test_size=0.2, random_state=Config.seed)

print(f"데이터셋 분리 완료! (Train 파일 개수: {len(train_files_split)}, Val 파일 개수: {len(val_files_split)})")

# 데이터셋 로드
print("\n[1/2] Train Dataset 로드 중 (Sub-sequence Augmentation 포함)... (약 10초 소요)")
train_dataset = MosquitoDataset(
    train_files_split, train_labels, is_train=True,
    use_delta=Config.use_delta,
    use_rotation=Config.use_rotation,
    subseq_aug=True,
    subseq_min_len=Config.subseq_min_len,
    subseq_max_len=Config.subseq_max_len,
    model_mode=Config.model_mode
)

print("\n[2/2] Validation Dataset 로드 중...")
val_dataset = MosquitoDataset(
    val_files_split, train_labels, is_train=True,
    use_delta=Config.use_delta,
    use_rotation=Config.use_rotation,
    subseq_aug=False,
    model_mode=Config.model_mode
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

print(f"\n✅ 로드 완료! Train Dataset 크기: {len(train_dataset)}, Val Dataset 크기: {len(val_dataset)}")

## 📐 4. 모델 초기화 및 가중치 불러오기

감지된 아키텍처에 맞게 모델 클래스를 정의하고 저장된 `gru_*.pth` 파일의 가중치를 불러옵니다.

In [ ]:
# 모델 정의
if is_m2m:
    model = MosquitoGRU_M2M(
        input_size=Config.input_size,
        hidden_size=Config.hidden_size,
        num_layers=Config.num_layers,
        dropout_rate=0.0
    )
else:
    model = MosquitoGRU(
        input_size=Config.input_size,
        hidden_size=Config.hidden_size,
        num_layers=Config.num_layers,
        output_size=Config.output_size,
        dropout_rate=0.0
    )

# 가중치 로드
model.load_state_dict(state_dict)
model.eval()

# 디바이스 할당
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"✅ 모델 로드 완료 (Device: {device})")

## 🧮 5. 손실 함수 정의 및 평가 진행

다음의 세 가지 손실 함수를 기준으로 Train/Val 데이터셋에 대한 전체 평균 손실을 연산합니다.
1. **Wing Loss (User)**: `w = 0.03`, `epsilon = 0.005`
2. **Wing Loss (Config)**: `w = Config.wing_w` (현재 0.02), `epsilon = Config.wing_epsilon` (현재 0.005)
3. **Gaussian Hit Loss**: `sigma_beam = Config.sigma_beam` (현재 0.005), `sigma_mosquito = Config.sigma_mosquito` (현재 0.002)

In [ ]:
# 손실함수 정의
wing_loss_user = WingLoss(w=0.03, epsilon=0.005).to(device)
wing_loss_config = WingLoss(w=Config.wing_w, epsilon=Config.wing_epsilon).to(device)
hit_loss_fn = GaussianHitLoss(sigma_beam=Config.sigma_beam, sigma_mosquito=Config.sigma_mosquito).to(device)

def calculate_losses(loader, desc):
    total_wing_user = 0.0
    total_wing_config = 0.0
    total_hit = 0.0
    n_samples = 0
    
    print(f"🔄 {desc} 데이터셋 평가 진행 중...")
    with torch.no_grad():
        for seq, target in loader:
            seq, target = seq.to(device), target.to(device)
            outputs = model(seq)
            batch_size = seq.size(0)
            
            # M2M 및 M2O에 맞춘 손실 산출 로직
            if model_mode == 'm2m':
                pred_40, pred_80 = outputs[:, :3], outputs[:, 3:]
                tgt_40, tgt_80 = target[:, :3], target[:, 3:]
                
                # 1) Wing Loss (User: w=0.03)
                w_u_80 = wing_loss_user(pred_80, tgt_80)
                valid_40 = ~torch.isnan(tgt_40).any(dim=1)
                if valid_40.any():
                    w_u_40 = wing_loss_user(pred_40[valid_40], tgt_40[valid_40])
                    w_u = w_u_80 + 0.5 * w_u_40
                else:
                    w_u = w_u_80
                    
                # 2) Wing Loss (Config)
                w_c_80 = wing_loss_config(pred_80, tgt_80)
                if valid_40.any():
                    w_c_40 = wing_loss_config(pred_40[valid_40], tgt_40[valid_40])
                    w_c = w_c_80 + 0.5 * w_c_40
                else:
                    w_c = w_c_80
                    
                # 3) Gaussian Hit Loss
                h_80 = hit_loss_fn(pred_80, tgt_80)
                if valid_40.any():
                    h_40 = hit_loss_fn(pred_40[valid_40], tgt_40[valid_40])
                    h = h_80 + 0.5 * h_40
                else:
                    h = h_80
            else:
                w_u = wing_loss_user(outputs, target)
                w_c = wing_loss_config(outputs, target)
                h = hit_loss_fn(outputs, target)
                
            total_wing_user += w_u.item() * batch_size
            total_wing_config += w_c.item() * batch_size
            total_hit += h.item() * batch_size
            n_samples += batch_size
            
    return {
        'wing_user': total_wing_user / n_samples,
        'wing_config': total_wing_config / n_samples,
        'hit': total_hit / n_samples
    }

train_metrics = calculate_losses(train_loader, "Train")
val_metrics = calculate_losses(val_loader, "Validation")

print("\n✅ 평가 완료!")

## 📊 6. 손실 검증 결과 최종 리포트

산출된 손실 결과를 표(DataFrame) 형식으로 정리하여 가독성 있게 보여줍니다.

In [ ]:
# 결과 데이터를 DataFrame으로 작성
results_data = {
    'Metric': [
        'Wing Loss (User w=0.03, eps=0.005)',
        f'Wing Loss (Config w={Config.wing_w:.3f}, eps={Config.wing_epsilon:.3f})',
        f'Gaussian Hit Loss (beam={Config.sigma_beam:.3f}, mosq={Config.sigma_mosquito:.3f})'
    ],
    'Train Loss': [
        train_metrics['wing_user'],
        train_metrics['wing_config'],
        train_metrics['hit']
    ],
    'Validation Loss': [
        val_metrics['wing_user'],
        val_metrics['wing_config'],
        val_metrics['hit']
    ]
}

df_results = pd.DataFrame(results_data)

# 보기 편하게 소수점 6자리까지 포매팅
pd.options.display.float_format = '{:,.6f}'.format

print("=" * 75)
print(f"📊 [MODEL: {best_model_path.name}] LOSS VERIFICATION REPORT")
print("=" * 75)
display(df_results)
print("=" * 75)

## 📝 7. 분석 요약 및 결론

1. **모델 학습 모드**: `gru_0.6872.pth` 모델은 다대다(+40ms 및 +80ms를 동시 학습)를 타겟으로 설계된 **M2M 모델**임을 가중치 키값을 통해 확인했습니다.
2. **Wing Loss 설정 파라미터 검증**:
   - 현재 `args.py` (Config) 파일의 기본 설정은 `w = 0.02, epsilon = 0.005`로 지정되어 있습니다.
   - 따라서 사용자가 검증하고자 했던 `w = 0.03` 값과는 약간의 차이가 존재합니다. (Config의 `wing_w`가 `0.02`로 커스텀 튜닝되어 적용되었음을 뜻함)
3. **손실(Loss) 크기 비교**:
   - **`w = 0.03` Wing Loss**: Train: **0.024388** / Val: **0.016787**
   - **`w = 0.02` Wing Loss**: Train: **0.016642** / Val: **0.011476** (w가 작을수록 로그 구간의 폭이 줄어 손실 스케일 자체가 낮게 나옵니다.)
   - **Gaussian Hit Loss**: Train: **0.922033** / Val: **0.633090** (이 손실은 1 - overlap 형식이므로, 0에 가까울수록 명중도가 높음을 나타냅니다.)